# 01b – Understanding the parsing pipeline, step by step

This notebook explains **every function in `01_parse_documents.ipynb`** by running it on small inputs and on the real data, so you can **see what each step does to the output of the step before it**.

**How to use it**
- Run the cells **top to bottom**. Each function is defined again here, right before its tests, so this notebook does not depend on notebook 01.
- Every code cell has a markdown cell **above** it (what we are about to do and why) and often one **below** it (how to read the output).
- **Nothing is written to `data/`** and **no paid API call is made** (the diagram step reads the existing cache). It is safe to run as often as you like.

**The journey of one document**

```
Word file (.docx)  ─►  XML body  ─►  blocks (paragraphs + tables, in order)
   ─►  each block classified (heading / list / para / table)
   ─►  table of contents removed  ─►  list runs demoted
   ─►  Markdown text  +  sections [{heading, level, text}]
Diagram image  ─►  bytes  ─►  fingerprint (cache key)  ─►  base64 data URL  ─►  text (vision LLM, cached)
```

We follow **P06 (Rebate Validation)** for most steps, **P07** for the table of contents, plus tiny hand-made documents to test edge cases.

## Step 1 – Setup: imports and paths

**What:** load the libraries and find the project folders.

- `ROOT` = project root, whether Jupyter was started in `notebooks/` or in the root.
- `sys.path.append(...)` lets Python find our own package `src` (so `from src.common.config import ...` works).
- `RAW`, `PROCESSED` and `ask` come from `src/common/config.py` (which also loads `.env`).

In [1]:
import sys, re, json, base64, hashlib
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
from src.common.config import RAW, PROCESSED, ask
from docx import Document
from docx.table import Table
from docx.text.paragraph import Paragraph
from docx.oxml.ns import qn
from docx.shared import Pt
import pandas as pd
pd.set_option("display.max_colwidth", 80)

print("Current folder :", Path.cwd())
print("Project root   :", ROOT)
print("RAW folder     :", RAW)
print("Last entry added to sys.path:", sys.path[-1])

Current folder : c:\Users\Dhruba\OneDrive\Desktop\test_agent\notebooks
Project root   : c:\Users\Dhruba\OneDrive\Desktop\test_agent
RAW folder     : C:\Users\Dhruba\OneDrive\Desktop\test_agent\data\raw
Last entry added to sys.path: c:\Users\Dhruba\OneDrive\Desktop\test_agent


**Read the output:** `Project root` must be your `test_agent` folder. If it shows `...\notebooks`, Jupyter was started somewhere unexpected.

### 1b – Finding the projects: `RAW.glob("*/*")`

`glob("*/*")` returns everything **exactly two levels** below `data/raw` → `<split>/<project>`. We first look at the **unfiltered** result, then keep only folders with `is_dir()`, then sort.

In [11]:
everything = list(RAW.glob("*/*"))
print(everything)
print("-"*100)

print("Unfiltered glob result (", len(everything), "items ):")
for p in everything[:len(everything)]:
    print("  ", "DIR " if p.is_dir() else "FILE", p.relative_to(RAW))
print("-"*100)
projects = sorted(p for p in RAW.glob("*/*") if p.is_dir())
print("\nAfter is_dir() + sorted():")
for p in projects:
    print("  ", p.parent.name.ljust(11), p.name)

[WindowsPath('C:/Users/Dhruba/OneDrive/Desktop/test_agent/data/raw/test/P08_Warranty_Returns'), WindowsPath('C:/Users/Dhruba/OneDrive/Desktop/test_agent/data/raw/test/P09_Vendor_Compliance'), WindowsPath('C:/Users/Dhruba/OneDrive/Desktop/test_agent/data/raw/train/P00_Pricing_Tool_UserSample'), WindowsPath('C:/Users/Dhruba/OneDrive/Desktop/test_agent/data/raw/train/P01_Invoice_Matching'), WindowsPath('C:/Users/Dhruba/OneDrive/Desktop/test_agent/data/raw/train/P02_Quote_Assistant'), WindowsPath('C:/Users/Dhruba/OneDrive/Desktop/test_agent/data/raw/train/P03_Contract_Intelligence'), WindowsPath('C:/Users/Dhruba/OneDrive/Desktop/test_agent/data/raw/train/P04_Product_Data_Enrichment'), WindowsPath('C:/Users/Dhruba/OneDrive/Desktop/test_agent/data/raw/train/P05_Freight_Claims'), WindowsPath('C:/Users/Dhruba/OneDrive/Desktop/test_agent/data/raw/validation/P06_Rebate_Validation'), WindowsPath('C:/Users/Dhruba/OneDrive/Desktop/test_agent/data/raw/validation/P07_Email_Order_Entry')]
------------

**Read the output:** any `FILE` lines in the first list (e.g. a stray file) disappear in the second list. `sorted` orders by full path, which is why `test` comes before `train` before `validation`. `.ljust(11)` pads the split name so the columns line up.

## Step 2 – The raw files of one project

**What:** look at what notebook 01 starts from. Every project folder has three files: BRD (input), architecture diagram (input), reference test plan (the "answer").

In [12]:
P06 = next(p for p in projects if p.name.startswith("P06"))
for f in sorted(P06.iterdir()):
    print(f"{f.name:45} {f.stat().st_size/1024:8.1f} KB")
brd_path = next(P06.glob("*_BRD.docx"))

P06_Rebate_Validation_Architecture.png           241.1 KB
P06_Rebate_Validation_BRD.docx                    39.2 KB
P06_Rebate_Validation_TestPlan.docx               40.3 KB


## Step 3 – Opening a Word file, and why the "easy" way loses information

python-docx offers `doc.paragraphs` and `doc.tables`. They are **two separate lists**, so you cannot tell **where** a table was in the document. Let's prove it.

In [13]:
doc = Document(brd_path)
print("doc.paragraphs :", len(doc.paragraphs), "paragraphs")
print("doc.tables     :", len(doc.tables), "tables")
print("\nFirst table, first row:", [c.text for c in doc.tables[0].rows[0].cells] if doc.tables else "(no tables)")
print("Question: which paragraph comes right before that table?  -> the two lists cannot tell us.")

doc.paragraphs : 103 paragraphs
doc.tables     : 4 tables

First table, first row: ['Phase', 'Scope']
Question: which paragraph comes right before that table?  -> the two lists cannot tell us.


### 3b – Inside the file: the XML body

A `.docx` is a zip of XML files. The text lives in `<w:body>`. Its **direct children, in order**, are paragraphs `<w:p>`, tables `<w:tbl>` and page settings `<w:sectPr>`.

`child.tag` holds the **full** XML name (namespace in curly braces). `qn("w:p")` turns the short name into that full name, so we can compare them.

In [18]:
print('qn("w:p")   =', qn("w:p"))
print('qn("w:tbl") =', qn("w:tbl"))
print()
children = list(doc.element.body.iterchildren())

first_tbl = next((i for i, c in enumerate(children) if c.tag == qn("w:tbl")), 0)
show_idx = list(range(0, 6)) + list(range(max(first_tbl - 3, 6), min(first_tbl + 3, len(children)))) + [len(children) - 1]
for i in show_idx:
    child = children[i]
    short = child.tag.split("}")[1]                      # drop the {namespace} part for display
    if child.tag == qn("w:p"):
        text = Paragraph(child, doc).text[:80]           # wrap just to read the visible text
    elif child.tag == qn("w:tbl"):
        text = "(table)"
    else:
        text = "(page settings)"
    print(f"{i:2}  <w:{short:6}>  {text}")

qn("w:p")   = {http://schemas.openxmlformats.org/wordprocessingml/2006/main}p
qn("w:tbl") = {http://schemas.openxmlformats.org/wordprocessingml/2006/main}tbl

 0  <w:p     >  Business Requirement Specification – Supplier Rebate Claim Validation
 1  <w:p     >  Version: 1.0
 2  <w:p     >  Date: 17 Feb 2026
 3  <w:p     >  Business Owner: Supplier Programs Finance
 4  <w:p     >  Status: Approved
 5  <w:p     >  Section 1 – Purpose
14  <w:p     >  3. Cut claim preparation time from 3 days to 4 hours per supplier
15  <w:p     >  4. Provide auditable calculations
16  <w:p     >  Section 4 – Scope of Work
17  <w:tbl   >  (table)
18  <w:p     >  
19  <w:p     >  Section 5 – User Roles
107  <w:sectPr>  (page settings)


**Read the output:** we print the first 6 children, the children around the **first table**, and the very last child. The left number is the position in the document: the table (`<w:tbl>`) sits **between** paragraphs (`<w:p>`) – exactly the information `doc.paragraphs` / `doc.tables` lose. The last child is `<w:sectPr>` (page settings), which we will skip.

## Step 4 – `iter_blocks(doc)`: paragraphs and tables in true order

**How:** walk the body children; wrap each `<w:p>` in a `Paragraph` object and each `<w:tbl>` in a `Table` object (so we get `.text`, `.style`, `.runs`, `.rows`). Anything else (`<w:sectPr>`) is skipped.

`yield` makes it a **generator**: blocks are produced one at a time when the caller loops over it.

In [17]:
def iter_blocks(doc):
    for child in doc.element.body.iterchildren():
        if child.tag == qn("w:p"):
            yield Paragraph(child, doc)
        elif child.tag == qn("w:tbl"):
            yield Table(child, doc)

gen = iter_blocks(doc)
print("iter_blocks returns:", type(gen).__name__, "(nothing has been read yet)")
first = next(gen)                                        # ask the generator for ONE block
print("First block:", type(first).__name__, "->", repr(first.text[:60]))

blocks_raw = list(iter_blocks(doc))                      # read all blocks
print("\nTotal blocks:", len(blocks_raw),
      "| paragraphs:", sum(isinstance(b, Paragraph) for b in blocks_raw),
      "| tables:", sum(isinstance(b, Table) for b in blocks_raw))
print()
for i, b in enumerate(blocks_raw[:14]):
    kind = "TABLE" if isinstance(b, Table) else "para "
    txt = b.rows[0].cells[0].text if isinstance(b, Table) else b.text
    print(f"{i:2} {kind} {txt[:70]!r}")

iter_blocks returns: generator (nothing has been read yet)
First block: Paragraph -> 'Business Requirement Specification – Supplier Rebate Claim V'

Total blocks: 107 | paragraphs: 103 | tables: 4

 0 para  'Business Requirement Specification – Supplier Rebate Claim Validation'
 1 para  'Version: 1.0'
 2 para  'Date: 17 Feb 2026'
 3 para  'Business Owner: Supplier Programs Finance'
 4 para  'Status: Approved'
 5 para  'Section 1 – Purpose'
 6 para  'Rebate Claim Validation will automate the calculation and validation o'
 7 para  'MVP1 reads rebate agreements, extracts tier rules, calculates earned r'
 8 para  'Section 2 – Current State'
 9 para  'Wesco manages over 400 supplier rebate agreements with different tiers'
10 para  'Agreement terms are in PDF contracts and amendments, and eligible prod'
11 para  'Section 3 – Objectives'
12 para  '1. Calculate earned rebates automatically for all active agreements'
13 para  '2. Recover unclaimed rebates (target +1.5% of rebate income)'


**Read the output:** the counts match `doc.paragraphs` + `doc.tables` from Step 3, but now tables appear **in their real position**. Empty paragraphs (`''`) are blank lines in Word; they are dropped later.

## Step 5 – `table_to_md(tbl)`: a Word table → a Markdown table

**How:**
1. For each row, take each cell's text, `strip()` it, replace newlines with spaces (a Markdown row must be one line) and `|` with `/` (`|` separates columns in Markdown).
2. Skip a row identical to the previous one (merged cells make python-docx repeat cells/rows).
3. Write the header row, the `|---|---|` separator, then the data rows.

In [19]:
def table_to_md(tbl):
    rows = []
    for r in tbl.rows:
        cells = [c.text.strip().replace("\n", " ").replace("|", "/") for c in r.cells]
        if not rows or cells != rows[-1]:
            rows.append(cells)
    if not rows:
        return ""
    out = ["| " + " | ".join(rows[0]) + " |", "|" + "---|" * len(rows[0])]
    out += ["| " + " | ".join(r) + " |" for r in rows[1:]]
    return "\n".join(out)

real_tables = [b for b in blocks_raw if isinstance(b, Table)]
if real_tables:
    t = real_tables[0]
    print("BEFORE – raw cell texts, row by row:")
    for r in t.rows[:4]:
        print("  ", [c.text for c in r.cells])
    print("\nAFTER – table_to_md():")
    print(table_to_md(t))
else:
    print("This BRD has no tables – see the hand-made test below.")

BEFORE – raw cell texts, row by row:
   ['Phase', 'Scope']
   ['In scope (MVP1)', 'Upload of rebate agreements (PDF), AI extraction of tiers, rates, eligible product lines and periods, analyst review of extracted terms, calculation from Delta Lake purchase data, claim worksheet with tier reached and earned amount, variance to accrual']
   ['In scope (MVP2)', 'Supplier portal submission, dispute tracking, accrual posting to finance, amendment comparison']
   ['Out of scope', 'Customer rebates (sell-side), price protection claims, supplier payment collection']

AFTER – table_to_md():
| Phase | Scope |
|---|---|
| In scope (MVP1) | Upload of rebate agreements (PDF), AI extraction of tiers, rates, eligible product lines and periods, analyst review of extracted terms, calculation from Delta Lake purchase data, claim worksheet with tier reached and earned amount, variance to accrual |
| In scope (MVP2) | Supplier portal submission, dispute tracking, accrual posting to finance, amendment comp

### 5b – Edge cases on a hand-made table

We build a tiny table that contains a **newline inside a cell**, a **`|` character**, a **merged cell** and a **duplicate row**, to watch each cleanup rule work.

In [20]:
t_doc = Document()
tbl = t_doc.add_table(rows=4, cols=2)
tbl.cell(0, 0).text, tbl.cell(0, 1).text = "Role", "Permission"
tbl.cell(1, 0).text, tbl.cell(1, 1).text = "Analyst", "Upload\nConfirm"          # newline inside a cell
tbl.cell(2, 0).text, tbl.cell(2, 1).text = "Manager", "Approve | Reject"         # pipe inside a cell
merged = tbl.cell(3, 0).merge(tbl.cell(3, 1)); merged.text = "Admin (merged across both columns)"

print("What python-docx returns per row (note the merged cell appears TWICE):")
for r in tbl.rows:
    print("  ", [c.text for c in r.cells])
print("\ntable_to_md():")
print(table_to_md(tbl))

What python-docx returns per row (note the merged cell appears TWICE):
   ['Role', 'Permission']
   ['Analyst', 'Upload\nConfirm']
   ['Manager', 'Approve | Reject']
   ['Admin (merged across both columns)', 'Admin (merged across both columns)']

table_to_md():
| Role | Permission |
|---|---|
| Analyst | Upload Confirm |
| Manager | Approve / Reject |
| Admin (merged across both columns) | Admin (merged across both columns) |


**Read the output:** `Upload\nConfirm` became `Upload Confirm`, `Approve | Reject` became `Approve / Reject` (otherwise Markdown would see 3 columns). The merged cell shows its text in both columns – that is how python-docx represents merges.

## Step 6 – `body_font_size(doc)`: the size of normal text

Every Word document has a style called **"Normal"** (default body text). Its size is a `Length` object, or `None` if not set explicitly (then Word uses its default). We return points (`.pt`), falling back to 11.

**Why:** later, "is this line bigger than normal text?" is a heading signal.

In [21]:
def body_font_size(doc):
    sz = doc.styles["Normal"].font.size
    return sz.pt if sz else 11.0

print("P06 Normal style size, raw value :", doc.styles["Normal"].font.size)
print("P06 body_font_size()             :", body_font_size(doc))
print("Brand-new empty Document()       :", body_font_size(Document()), "(not set -> fallback 11.0)")
base = body_font_size(doc)

P06 Normal style size, raw value : 139700
P06 body_font_size()             : 11.0
Brand-new empty Document()       : 11.0 (not set -> fallback 11.0)


**Read the output:** the raw value is stored in **EMU** (English Metric Units, Word's internal unit: 12,700 EMU = 1 pt), so `139700` = 11 pt. `.pt` does this conversion for us.

## Step 7 – `NUM_RE`, `number_level`, `strip_number`: understanding numbering

`NUM_RE` recognises numbering **at the start** of a line. It has four alternatives, each a **named group**:

| group | matches | examples |
|---|---|---|
| `sec` | `section` + number + dash/colon | `Section 3 – `, `section 12: ` |
| `dec` | decimal numbers + optional `.` or `)` + space | `4. `, `2.3 `, `1.2.1 `, `4) ` |
| `rom` | Roman numerals + `.` + space | `IV. `, `xii. ` |
| `let` | one letter + `)` + space | `b) ` |

`re.I` = ignore upper/lower case.
- `number_level`: `dec` → number of dots + 1; `let` → 2; `sec`/`rom` → 1; no numbering → `None`.
- `strip_number`: removes the numbering and trims spaces.

In [24]:
NUM_RE = re.compile(r"^\s*(?:(?P<sec>section\s+\d+\s*[–—:-]\s*)|(?P<dec>\d+(?:\.\d+)*)[.)]?\s+|(?P<rom>[IVXLC]+)\.\s+|(?P<let>[a-z])\)\s+)", re.I)

def strip_number(text):
    return NUM_RE.sub("", text).strip()

def number_level(text):
    m = NUM_RE.match(text)
    if not m:
        return None
    if m.group("dec"):
        return m.group("dec").count(".") + 1
    if m.group("let"):
        return 2
    return 1

samples = ["Section 3 – Purpose", "4. Scope", "2.3 Test levels", "1.2.1 Upload", "4) Risks", "IV. Requirements",
           "b) Dependencies", "Introduction", "2026 Roadmap", "MVP1: Upload", "3.Scope"]
rows = []
for s in samples:
    m = NUM_RE.match(s)
    group = next((g for g, v in m.groupdict().items() if v), None) if m else None
    rows.append({"text": s, "matched part": repr(m.group(0)) if m else "-", "group": group,
                 "number_level": number_level(s), "strip_number": strip_number(s)})
pd.DataFrame(rows)

,text,matched part,group,number_level,strip_number
0,Section 3 – Purpose,'Section 3 – ',sec,1.0,Purpose
1,4. Scope,'4. ',dec,1.0,Scope
2,2.3 Test levels,'2.3 ',dec,2.0,Test levels
3,1.2.1 Upload,'1.2.1 ',dec,3.0,Upload
4,4) Risks,'4) ',dec,1.0,Risks
5,IV. Requirements,'IV. ',rom,1.0,Requirements
6,b) Dependencies,'b) ',let,2.0,Dependencies
7,Introduction,-,NaN,NaN,Introduction
8,2026 Roadmap,'2026 ',dec,1.0,Roadmap
9,MVP1: Upload,-,NaN,NaN,MVP1: Upload


**Read the output – things to notice:**
- `Introduction`, `MVP1: Upload` → no match (no numbering at the **start**).
- `3.Scope` → **no match**, because the pattern requires a **space** after the number. Real documents almost always have one.
- `2026 Roadmap` → matches `dec` (a year looks like a number). That is acceptable: a numbered line only becomes a heading if other rules agree (Step 11).

## Step 8 – `BULLET_RE`: typed bullets

Matches a line that **starts** with a bullet character (`• ▪ ◦ ● ○ ■ □ ➢ ► - – — *`) or with `Step <n>:` / `Step <n>.` / `Step <n>)`.

`BULLET_RE.sub("", text)` removes the bullet – used later when we write list items.

In [25]:
BULLET_RE = re.compile(r"^\s*(?:[•▪◦●○■□➢►\-–—*]|step\s+\d+\s*[:.)])\s*", re.I)

for s in ["• First bullet", "– No disputes", "- dash item", "* star item", "Step 2: Upload file",
          "step 10) review", "1. Numbered item", "Stepwise approach", "-5% discount"]:
    m = BULLET_RE.match(s)
    print(f"{s!r:25} match={bool(m)!s:5}  after sub -> {BULLET_RE.sub('', s)!r}")

'• First bullet'          match=True   after sub -> 'First bullet'
'– No disputes'           match=True   after sub -> 'No disputes'
'- dash item'             match=True   after sub -> 'dash item'
'* star item'             match=True   after sub -> 'star item'
'Step 2: Upload file'     match=True   after sub -> 'Upload file'
'step 10) review'         match=True   after sub -> 'review'
'1. Numbered item'        match=False  after sub -> '1. Numbered item'
'Stepwise approach'       match=False  after sub -> 'Stepwise approach'
'-5% discount'            match=True   after sub -> '5% discount'


**Read the output:** `1. Numbered item` is **not** a bullet (numbers are handled by `NUM_RE`). `Stepwise approach` is not matched because `step` must be followed by a space and a number. `-5% discount` **is** matched – a leading hyphen counts as a bullet; acceptable because such lines are rare in BRDs.

## Step 9 – `run_props(para)`: reading formatting

A paragraph is made of **runs** = stretches of text with identical formatting. `"**Scope:** in MVP1"` is 2 runs.

`run_props` returns `(bold, underline, largest_font_size)`:
- `bold` is True only if **all** visible runs are bold (one bold word does not make a bold line).
- font size is only known if set explicitly on a run; otherwise `None`.

In [26]:
def run_props(para):
    runs = [r for r in para.runs if r.text.strip()]
    if not runs:
        return False, False, None
    bold = all(r.bold or (r.style is not None and r.style.font.bold) for r in runs)
    under = all(bool(r.underline) for r in runs)
    sizes = [r.font.size.pt for r in runs if r.font.size]
    return bold, under, (max(sizes) if sizes else None)

f_doc = Document()
def para_with_runs(*parts):
    # parts: (text, bold, underline, size_pt)
    p = f_doc.add_paragraph()
    for text, b, u, sz in parts:
        r = p.add_run(text); r.bold = b; r.underline = u
        if sz: r.font.size = Pt(sz)
    return p

tests = {
    "plain text":              para_with_runs(("Plain text", None, None, None)),
    "fully bold":              para_with_runs(("Implementation", True, None, None)),
    "partly bold":             para_with_runs(("Scope: ", True, None, None), ("in MVP1", None, None, None)),
    "underlined":              para_with_runs(("Overview", None, True, None)),
    "bold + 14pt":             para_with_runs(("Introduction", True, None, 14)),
}
for name, p in tests.items():
    runs_view = [(r.text, r.bold, r.underline, r.font.size.pt if r.font.size else None) for r in p.runs]
    print(f"{name:12} runs={runs_view}\n{'':12} run_props -> {run_props(p)}\n")

plain text   runs=[('Plain text', None, None, None)]
             run_props -> (False, False, None)

fully bold   runs=[('Implementation', True, None, None)]
             run_props -> (True, False, None)

partly bold  runs=[('Scope: ', True, None, None), ('in MVP1', None, None, None)]
             run_props -> (False, False, None)

underlined   runs=[('Overview', None, True, None)]
             run_props -> (False, True, None)

bold + 14pt  runs=[('Introduction', True, None, 14.0)]
             run_props -> (True, False, 14.0)



**Read the output:** "partly bold" gives `bold=False` because not **all** runs are bold. The size is `None` unless a run sets it (14.0 in the last case).

## Step 10 – `is_indented(para)`

Left indent ≥ 9 pt counts as indented (lists are often indented, headings usually are not).

In [ ]:
def is_indented(para):
    ind = para.paragraph_format.left_indent
    return ind is not None and ind.pt >= 9

i_doc = Document()
for indent in [None, 4, 9, 21.6]:
    p = i_doc.add_paragraph("Some item")
    if indent is not None:
        p.paragraph_format.left_indent = Pt(indent)
    print(f"left_indent={indent!s:5} pt -> is_indented = {is_indented(p)}")

## Step 11 – `classify(para, base_size)`: heading, list item or paragraph?

This is the core decision. It returns `(kind, level, strong)`; `strong=True` means "certain" (Word heading style).

**The rules are checked in this order – the first rule that fires wins:**

| # | Rule | Result |
|---|---|---|
| 1 | empty text | `None` (skip) |
| 2 | Word style "Heading N" / "Title" | heading level N, **strong** |
| 3 | Word list style or Word auto-numbering | list |
| 4 | starts with a typed bullet / `Step n:` (and not numbering) | list |
| 5 | indented and not big | list |
| 6 | long (> 90 chars or > 12 words) **or** ends like a sentence (`.` `;` `,`) | para |
| 7 | bold and ends with `:` | heading level 3 (label) |
| 8 | ends with `:` (not bold) | para (introduces a list) |
| 9 | bold / underlined / ALL CAPS / bigger font | heading (level from number, else 1 for big/caps/underline, 2 for bold) |
| 10 | numbered, ≤ 8 words, capital letter after the number | heading (weak) |
| 11 | anything else | para |

**Why lists before headings:** without rules 3–5, short list lines like "– No disputes" or "1. Provide auditable calculations" were classified as headings (P00 had 127 sections, 79 empty).

In [ ]:
def classify(para, base_size):
    text = para.text.strip()
    if not text:
        return None, None, False
    style = para.style.name if para.style is not None else ""
    if style.lower().startswith("heading"):
        return "heading", int(re.sub(r"\D", "", style) or 1), True
    if style.lower().startswith("title"):
        return "heading", 1, True
    if "list" in style.lower() or (para._p.pPr is not None and para._p.pPr.numPr is not None):
        return "list", None, False
    if BULLET_RE.match(text) and not NUM_RE.match(text):
        return "list", None, False
    bold, under, size = run_props(para)
    big = size is not None and size >= base_size + 1.5
    if is_indented(para) and not big:
        return "list", None, False
    words = len(text.split())
    short = len(text) <= 90 and words <= 12
    ends_sentence = text.endswith((".", ";", ",")) and not re.search(r"\b(e\.g|etc)\.$", text)
    letters = re.sub(r"[^A-Za-z]", "", text)
    caps = len(letters) >= 4 and letters.isupper()
    numbered = number_level(text)
    if not short or ends_sentence:
        return "para", None, False
    if bold and text.endswith(":"):
        return "heading", 3, False
    if text.endswith(":"):
        return "para", None, False
    if bold or under or caps or big:
        if numbered:
            return "heading", min(numbered, 3), False
        return "heading", (1 if (big or caps or under) else 2), False
    if numbered and words <= 8 and strip_number(text)[:1].isupper():
        return "heading", min(numbered, 3), False
    return "para", None, False

### 11b – A helper that shows the **inputs** of the decision

`features()` computes the same signals `classify` looks at, so you can see **why** each line got its result.

In [ ]:
def features(para, base_size):
    text = para.text.strip()
    bold, under, size = run_props(para)
    letters = re.sub(r"[^A-Za-z]", "", text)
    return {
        "text": text[:45],
        "style": para.style.name if para.style is not None else "",
        "bullet": bool(BULLET_RE.match(text)),
        "indent": is_indented(para),
        "words": len(text.split()),
        "ends_sentence": text.endswith((".", ";", ",")),
        "bold": bold, "under": under,
        "big": size is not None and size >= base_size + 1.5,
        "caps": len(letters) >= 4 and letters.isupper(),
        "numbered": number_level(text),
        "-> RESULT": classify(para, base_size),
    }

### 11c – Test on hand-made lines covering every rule

In [ ]:
c_doc = Document()
def add(text, bold=None, under=None, size=None, style=None, indent=None):
    p = c_doc.add_paragraph(style=style) if style else c_doc.add_paragraph()
    r = p.add_run(text); r.bold = bold; r.underline = under
    if size: r.font.size = Pt(size)
    if indent: p.paragraph_format.left_indent = Pt(indent)
    return p

cases = [
    add("Scope of Testing", style="Heading 2"),                         # rule 2
    add("Upload agreements", style="List Bullet"),                      # rule 3
    add("– No disputes"),                                                # rule 4
    add("Step 2: Upload file"),                                          # rule 4
    add("Indented item", indent=21.6),                                   # rule 5
    add("The system must validate every uploaded file before processing."),  # rule 6 (sentence)
    add("Dependencies:", bold=True),                                     # rule 7
    add("The system must:"),                                             # rule 8
    add("PROJECT OVERVIEW"),                                             # rule 9 caps
    add("Implementation", bold=True),                                    # rule 9 bold
    add("2.1 Test Levels", bold=True),                                   # rule 9 numbered
    add("Introduction", size=14),                                        # rule 9 big
    add("4. Scope"),                                                     # rule 10
    add("4. reduce cost"),                                               # rule 11 (lowercase after number)
    add("API"),                                                          # rule 11 (caps needs >= 4 letters)
]
pd.DataFrame([features(p, 11.0) for p in cases])

**Read the output:** follow each row left to right and find the first rule that applies. E.g. `4. reduce cost` is numbered but the first letter after the number is lowercase → para; `API` is all caps but has only 3 letters → para.

### 11d – Test on the real P06 BRD (first 40 blocks)

In [ ]:
real_rows = [features(b, base) for b in blocks_raw[:40] if isinstance(b, Paragraph) and b.text.strip()]
pd.DataFrame(real_rows)[["text", "bold", "under", "caps", "numbered", "words", "-> RESULT"]]

**Read the output:** the "Section N – …" lines are headings (bold + underlined + numbered). The numbered lines under them ("1. Calculate earned rebates…") are already `list` – not because of their numbers, but because they are **indented** (rule 5). In documents where numbered list lines are *not* indented, rule 10 would make them weak headings – that is the case the next step (`demote_runs`) catches.

## Step 12 – `demote_runs(blocks)`: the second safety net

**Idea:** real headings always have content between them. Two **weak** (not Word-style) headings of the **same level**, directly next to each other, are really a numbered list. Blank blocks (`kind=None`) are ignored when checking "next to each other". Level 3 (bold labels) is excluded.

In [ ]:
def demote_runs(blocks):
    idx = [i for i, b in enumerate(blocks) if b["kind"] is not None]
    for a, b in zip(idx, idx[1:]):
        x, y = blocks[a], blocks[b]
        if (x["kind"] == "heading" and y["kind"] == "heading" and not x["strong"] and not y["strong"]
                and x["level"] == y["level"] and x["level"] < 3):
            x["demote"] = y["demote"] = True
    for b in blocks:
        if b.get("demote"):
            b["kind"], b["level"] = "list", None
    return blocks

toy = [
    {"kind": "heading", "level": 1, "strong": False, "text": "Objectives"},
    {"kind": "para",    "level": None, "strong": False, "text": "The goals are:"},
    {"kind": "heading", "level": 1, "strong": False, "text": "1. Reduce cost"},
    {"kind": None,      "level": None, "strong": False, "text": ""},              # blank line – ignored
    {"kind": "heading", "level": 1, "strong": False, "text": "2. Faster quotes"},
    {"kind": "heading", "level": 1, "strong": True,  "text": "Scope"},             # Word style – never demoted
    {"kind": "para",    "level": None, "strong": False, "text": "MVP1 covers upload."},
]
before = [(b["text"], b["kind"]) for b in toy]
demote_runs(toy)
pd.DataFrame({"text": [t for t, _ in before], "before": [k for _, k in before], "after": [b["kind"] for b in toy]})

**Read the output:** "1. Reduce cost" and "2. Faster quotes" are adjacent weak level-1 headings (the blank line between them does not count) → both become `list`. "Objectives" is followed by a paragraph, and "Scope" is a Word-style heading, so they stay headings.

Also see `zip(idx, idx[1:])` in action – it produces **consecutive pairs**:

In [ ]:
idx = [0, 1, 2, 4, 5, 6]
print(list(zip(idx, idx[1:])))

## Step 13 – Table of contents helpers: `TOC_RE` and `norm`

- `TOC_RE` matches a line that is **exactly** "Contents" or "Table of Contents" (any case).
- `norm()` builds a comparison key (no numbering, lowercase, only letters/digits/spaces), so a TOC entry can be matched with the real heading later in the body.

In [ ]:
TOC_RE = re.compile(r"^(table of contents|contents)$", re.I)

def norm(s):
    return re.sub(r"[^a-z0-9 ]", "", strip_number(s).lower()).strip()

for s in ["Contents", "TABLE OF CONTENTS", "1. Contents:", "Contents of the BRD"]:
    print(f"{s!r:25} TOC_RE on strip_number(...).rstrip(':') -> {bool(TOC_RE.match(strip_number(s).rstrip(':')))}")
print()
for a, b in [("1. Purpose", "Purpose:"), ("Section 2 – Current State", "Current state"), ("Purpose ...... 3", "Purpose")]:
    print(f"norm({a!r}) = {norm(a)!r:22} norm({b!r}) = {norm(b)!r:18} equal: {norm(a) == norm(b)}")

**Read the output:** "Contents of the BRD" is **not** a TOC title (it must be the whole line). The last pair shows a **known limitation**: a TOC entry with page numbers (`Purpose ...... 3`) does not match `Purpose`. Our data has no page numbers; real Word TOCs often do (an easy future fix).

## Step 14 – `parse_docx(path)`, pass by pass

The original `parse_docx` does four passes in one function. Here we split it into **four small functions** with **identical logic**, so we can look at the data after each pass. At the end we prove the result is identical to the files notebook 01 saved.

We use **P07**, whose BRD has a table of contents.

### Pass 1 – classify every block

Every block becomes a small dict `{kind, level, strong, text}`. Tables become Markdown (empty table → `kind=None`).

In [ ]:
def pass1_classify(doc):
    base = body_font_size(doc)
    blocks = []
    for blk in iter_blocks(doc):
        if isinstance(blk, Table):
            md_t = table_to_md(blk)
            blocks.append({"kind": "table" if md_t else None, "level": None, "strong": False, "text": md_t})
        else:
            k, l, s = classify(blk, base)
            blocks.append({"kind": k, "level": l, "strong": s, "text": blk.text.strip()})
    return blocks

P07 = next(p for p in projects if p.name.startswith("P07"))
p07_doc = Document(next(P07.glob("*_BRD.docx")))
b1 = pass1_classify(p07_doc)
print("Blocks:", len(b1), "| kinds:", pd.Series([b["kind"] for b in b1]).value_counts(dropna=False).to_dict())
pd.DataFrame(b1[:25])

**Read the output:** `kind=None` rows are blank lines. Look for the **"Contents"** line and the short lines after it – those are the table of contents, still present after pass 1.

### Pass 2 – remove the table of contents

1. Find the "Contents" line, hide it (`kind=None`).
2. Collect the following short lines as TOC entries and hide them – **stop** at a table, a long line (≥ 90 chars) or when the **first TOC entry appears again** (that is where the body starts).
3. If that first body heading was not recognised as a heading, force it to be one (we know it is, because it matches the TOC).

`copy.deepcopy` is used only here, so you can compare before/after; the original function works in place.

In [ ]:
import copy

def pass2_skip_toc(blocks):
    toc, i = [], 0
    while i < len(blocks):
        b = blocks[i]
        if b["kind"] in ("heading", "para", "list") and TOC_RE.match(strip_number(b["text"]).rstrip(":")):
            b["kind"] = None; j = i + 1
            while j < len(blocks) and len(toc) < 40:
                t = blocks[j]
                if t["kind"] is None:
                    j += 1; continue
                if t["kind"] == "table" or len(t["text"]) >= 90 or (toc and norm(t["text"]) == norm(toc[0])):
                    break
                toc.append(t["text"]); t["kind"] = None; j += 1
            if j < len(blocks) and toc and norm(blocks[j]["text"]) == norm(toc[0]) and blocks[j]["kind"] != "heading":
                blocks[j]["kind"], blocks[j]["level"] = "heading", number_level(blocks[j]["text"]) or 1
            break
        i += 1
    return blocks, toc

b2, toc = pass2_skip_toc(copy.deepcopy(b1))
print("TOC entries found:", len(toc))
for t in toc:
    print("   ", t)
changed = [(i, b1[i]["text"][:50], b1[i]["kind"], b2[i]["kind"]) for i in range(len(b1)) if b1[i]["kind"] != b2[i]["kind"]]
pd.DataFrame(changed, columns=["block #", "text", "kind before", "kind after"])

**Read the output:** the "Contents" line and every TOC entry changed to `None` (hidden). If P07's first body heading had not been detected, it would show `kind after = heading` here.

### Pass 3 – demote list runs

In [ ]:
b3 = demote_runs(copy.deepcopy(b2))
changed = [(i, b2[i]["text"][:60], b2[i]["kind"], b3[i]["kind"]) for i in range(len(b2)) if b2[i]["kind"] != b3[i]["kind"]]
print("Blocks demoted from heading to list:", len(changed))
pd.DataFrame(changed, columns=["block #", "text", "kind before", "kind after"])

**Read the output – an honest side effect:** in P07 the rule demotes two lines in the Appendix ("… MVP1 Scope Overview" and "What's included"). They are really an appendix title directly followed by a sub-heading with the **same formatting**, so the rule cannot tell them apart from a list. This is the trade-off of a simple rule: it fixes many list items in other documents but occasionally turns a real heading into a list line. **No text is lost** – the lines stay in the Appendix section as `- …` items. A future improvement could skip demotion when the next block is followed by a paragraph.

### Pass 4 – build Markdown and sections

Walk the blocks:
- **heading** → close the current section, start a new one, write `#`×level + title (numbering and trailing `:` removed)
- **list** → remove bullet and numbering, write `- item`
- **para / table** → write as is
- everything before the first heading goes into a level-0 **"(front matter)"** section

`close()` saves the current section; it is a nested function that always sees the **latest** `cur`.

In [ ]:
def pass4_build(blocks):
    lines, sections = [], []
    cur = {"heading": "(front matter)", "level": 0, "text": []}
    def close():
        if cur["text"] or cur["level"] > 0:
            sections.append({"heading": cur["heading"], "level": cur["level"], "text": "\n".join(cur["text"]).strip()})
    for b in blocks:
        kind, text = b["kind"], b["text"]
        if kind is None:
            continue
        if kind == "heading":
            close()
            title = strip_number(text).rstrip(":").strip() or text
            cur = {"heading": title, "level": b["level"], "text": []}
            lines.append("\n" + "#" * b["level"] + " " + title)
        elif kind == "list":
            item = "- " + strip_number(BULLET_RE.sub("", text))
            lines.append(item); cur["text"].append(item)
        else:
            lines.append(text); cur["text"].append(text)
    close()
    return "\n".join(lines).strip(), sections

md_text, sections = pass4_build(b3)
print("----- first 40 lines of the Markdown -----")
print("\n".join(md_text.splitlines()[:40]))

In [ ]:
pd.DataFrame([{"level": s["level"], "heading": s["heading"], "words": len(s["text"].split()),
               "text (start)": s["text"][:70]} for s in sections])

### Putting it together: `parse_docx` = pass 1 → 2 → 3 → 4

This is the same function as in notebook 01, written as four calls. The check below compares our result with the files notebook 01 saved in `data/processed` – they must be **identical**.

In [ ]:
def parse_docx(path):
    doc = Document(path)
    blocks = pass1_classify(doc)
    blocks, toc = pass2_skip_toc(blocks)
    blocks = demote_runs(blocks)
    md_text, sections = pass4_build(blocks)
    return md_text, sections, toc

for proj in [P06, P07]:
    md_new, secs_new, _ = parse_docx(next(proj.glob("*_BRD.docx")))
    saved = PROCESSED / proj.parent.name / proj.name
    same_md = md_new == (saved / "brd.md").read_text(encoding="utf-8")
    same_secs = secs_new == json.loads((saved / "brd_sections.json").read_text(encoding="utf-8"))
    print(f"{proj.name:25} markdown identical: {same_md}   sections identical: {same_secs}")

**Read the output:** `True / True` proves the step-by-step version is exactly what notebook 01 does. (If you changed notebook 01 after saving the files, re-run its Cell 6 first.)

## Step 15 – `section_stats(secs)`: counting sections and empty ones

A section counts as **empty** only if it has **no text and no sub-sections** (the next section is not deeper). Parent headings like "Integrations" followed by H2s are fine.

In [ ]:
def section_stats(secs):
    body = [s for s in secs if s["level"] > 0]
    empty = 0
    for i, s in enumerate(body):
        nxt = body[i + 1] if i + 1 < len(body) else None
        has_children = nxt is not None and nxt["level"] > s["level"]
        if not s["text"] and not has_children:
            empty += 1
    return len(body), empty

toy_secs = [
    {"heading": "(front matter)", "level": 0, "text": "Version 1.0"},   # level 0 – not counted
    {"heading": "Integrations",   "level": 1, "text": ""},              # empty BUT has a child -> OK
    {"heading": "Identity",       "level": 2, "text": "Azure AD"},
    {"heading": "No disputes",    "level": 1, "text": ""},              # empty and no child -> EMPTY
    {"heading": "Scope",          "level": 1, "text": "MVP1 only"},
]
print("toy example  -> (sections, empty) =", section_stats(toy_secs))
print("P07 BRD      -> (sections, empty) =", section_stats(sections))

## Step 16 – `describe_arch(img_path)`: diagram → text, piece by piece (no API call)

The function: read bytes → fingerprint → if cached, return the cached text → otherwise build a base64 **data URL** and send it with the prompt to gpt-5-mini → save to cache.

Below we run each piece separately **without calling the API**.

In [ ]:
CACHE = PROCESSED / "_cache"
img_path = next(P06.glob("*_Architecture.*"))
data = img_path.read_bytes()
print("1. image file     :", img_path.name, "|", len(data), "bytes | first bytes:", data[:8])

full_hash = hashlib.sha1(data).hexdigest()
key = full_hash[:16]
print("2. SHA-1          :", full_hash, "(40 hex chars) -> key uses first 16:", key)

cache_file = CACHE / f"arch_{key}.md"
print("3. cache file     :", cache_file.name, "| exists:", cache_file.exists())

mime = "image/png" if img_path.suffix.lower() == ".png" else "image/jpeg"
b64 = base64.b64encode(data).decode()
print("4. mime type      :", mime)
print("5. base64 text    :", len(b64), "characters (about 4/3 of the byte size) | starts:", b64[:40], "...")
print("6. data URL starts:", f"data:{mime};base64,{b64[:30]}...")

**Read the output:**
- The PNG file starts with the bytes `\x89PNG` – every PNG does.
- Changing **one pixel** changes the SHA-1 completely → new cache key → new description. Renaming the file does **not** change it.
- Base64 makes the image about 33% bigger, but turns it into plain text that fits in a JSON request.

### 16b – Calling `describe_arch` (safe: only reads the cache)

`RUN_API = False` makes this cell refuse to call the API if the cache is missing. Set it to `True` only if you really want a new (paid) description.

⚠️ **Cache trap:** the cache key depends on the **image**, not on the prompt. If you change `ARCH_PROMPT`, delete `data/processed/_cache` to get new descriptions.

In [ ]:
ARCH_SYSTEM = ("You convert software architecture diagrams into precise text for test planning. "
               "Describe ONLY what is visible. Use the exact component and label names from the diagram.")
ARCH_PROMPT = '''Describe this solution architecture diagram using exactly these headings:
TITLE:
USERS / CLIENTS:
IDENTITY & ENTRY POINTS: (e.g. Azure AD, App Gateway)
APPLICATION SERVICES:
AI SERVICES:
DATA STORES:
INTEGRATION / MESSAGING:
EXTERNAL / ENTERPRISE SYSTEMS:
SUPPORTING SERVICES: (monitoring, CI/CD, secrets)
CONNECTIONS: one line per arrow -> source -> target | label (protocol / auth / purpose)
ENVIRONMENT NOTES: (subscription, non-production, network boundaries)'''

def describe_arch(img_path):
    data = img_path.read_bytes()
    key = hashlib.sha1(data).hexdigest()[:16]
    cache = CACHE / f"arch_{key}.md"
    if cache.exists():
        return cache.read_text(encoding="utf-8")
    mime = "image/png" if img_path.suffix.lower() == ".png" else "image/jpeg"
    b64 = base64.b64encode(data).decode()
    text = ask(ARCH_SYSTEM, [{"type": "input_text", "text": ARCH_PROMPT},
                             {"type": "input_image", "image_url": f"data:{mime};base64,{b64}"}], effort="low")
    cache.write_text(text, encoding="utf-8")
    return text

RUN_API = False
if cache_file.exists() or RUN_API:
    arch_text = describe_arch(img_path)
    print(arch_text[:1200])
else:
    print("No cache for this image and RUN_API=False -> skipped (no cost).")

**Read the output:** the same headings as `ARCH_PROMPT`. The **CONNECTIONS** lines (`source -> target | label`) are what later become integration and security tests.

## Step 17 – The "process all projects" loop, dry run for one project

Notebook 01's Cell 6 does, for every project: find the files → parse BRD and test plan → write 5 files → add a row to the quality report. Here we do it for **P06 without writing anything**, and compare our row with the saved `parse_report.csv`.

In [ ]:
def find(folder, pattern):
    hits = sorted(folder.glob(pattern))
    return hits[0] if hits else None

proj = P06
row = {"split": proj.parent.name, "project": proj.name}
for kind, pattern in [("brd", "*BRD*.docx"), ("testplan", "*TestPlan*.docx")]:
    f = find(proj, pattern)
    print(f"{kind:8} -> found file: {f.name if f else None}")
    md_k, secs_k, toc_k = parse_docx(f)
    row[f"{kind}_sections"], row[f"{kind}_empty"] = section_stats(secs_k)
    row[f"{kind}_words"] = len(md_k.split())
    row[f"{kind}_toc"] = len(toc_k) > 0
row["arch"] = "ok" if find(proj, "*Architecture.*") else "MISSING"

saved_report = pd.read_csv(PROCESSED / "parse_report.csv")
saved_row = saved_report[saved_report.project == proj.name].iloc[0].to_dict()
pd.DataFrame({"dry run (now)": row, "saved parse_report.csv": saved_row})

**Read the output:** both columns should be identical. The output files notebook 01 writes for this project are:

In [ ]:
out = PROCESSED / P06.parent.name / P06.name
for f in sorted(out.iterdir()):
    print(f"{f.name:25} {f.stat().st_size:7} bytes")

## Step 18 – `show()`: reading a processed file back

`json.loads` turns the saved JSON text back into a Python list of dicts; we print the first sections.

In [ ]:
def show(project_prefix, kind="testplan", n_sections=6):
    proj = next(p for p in (PROCESSED).glob("*/*") if p.name.startswith(project_prefix))
    secs = json.loads((proj / f"{kind}_sections.json").read_text(encoding="utf-8"))
    for s in secs[: n_sections]:
        print("=" * 90); print(f"[H{s['level']}] {s['heading']}"); print(s["text"][:600])

show("P06", "brd", 4)

## Summary – what each function does to the previous output

| Step | Function | Input | Output |
|---|---|---|---|
| 4 | `iter_blocks` | Word document | Paragraph / Table objects **in document order** |
| 5 | `table_to_md` | Table object | Markdown table text |
| 6 | `body_font_size` | Word document | normal text size (pt) |
| 7 | `NUM_RE`, `number_level`, `strip_number` | a line of text | numbering level / text without numbering |
| 8 | `BULLET_RE` | a line of text | is it a typed bullet? / text without bullet |
| 9–10 | `run_props`, `is_indented` | Paragraph | bold, underline, size / indented? |
| 11 | `classify` | Paragraph + body size | `(kind, level, strong)` |
| 14 pass 1 | `pass1_classify` | Word document | list of block dicts |
| 14 pass 2 | `pass2_skip_toc` | block dicts | block dicts without the TOC + the TOC entries |
| 12 / pass 3 | `demote_runs` | block dicts | adjacent weak headings turned into list items |
| 14 pass 4 | `pass4_build` | block dicts | **Markdown text + sections list** |
| 15 | `section_stats` | sections list | (number of sections, empty sections) |
| 16 | `describe_arch` | diagram image | diagram as structured text (cached) |
| 17 | process loop | project folder | 5 files in `data/processed` + one report row |

**Try it yourself:** in Step 11c, add your own line with `add("...", bold=True)` and predict the result before running.